# Anvil Framework — Jupyter Rich Display

Anvil provides HTML rich display for all major result types. In a Jupyter cell, just evaluate a Quantity, Result, SweepResult, or SensitivityResult to get a formatted HTML table automatically.

**Prerequisites:** `pip install jupyter` and open this notebook from the `examples/` directory.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import anvil
import numpy as np
from anvil import Q, System
from anvil import K, Pa, MPa, m, s, kg, J, kN, km

print(f'Anvil {anvil.__version__} loaded')

## 1. Quantity — HTML badge display

A `Quantity` renders as a styled inline badge showing the value and unit.

In [ ]:
# Evaluating a Quantity in a cell shows an HTML badge
T = Q(3500, 'K', name='chamber_temperature')
T

In [ ]:
# LaTeX repr (e.g. in math contexts)
from IPython.display import display, Math
display(Math(T._repr_latex_()))

In [ ]:
# Arithmetic preserves display
v    = 340 * (m/s)
rho  = Q(1.225, 'kg/m^3')
q_dyn = 0.5 * rho * v**2
q_dyn

## 2. Result — HTML table

A `Result` (returned by `solve_forward()` etc.) renders as a two-column HTML table separating **inputs** from **outputs**.

In [ ]:
# Solve a rocket nozzle and display the result
nozzle = anvil.S.rocket_nozzle.copy()
nozzle.set(
    P0=20e6,       # Pa  — LOX/LH2, 20 MPa chamber pressure
    T0=3500,       # K
    gamma=1.20,
    R_gas=520,     # J/kg/K
    A_throat=0.005,
    A_exit=0.08,
    P_amb=0,       # vacuum
)

result = nozzle.solve_forward()
result  # Renders as HTML table in Jupyter

In [ ]:
# Access specific results
print(f"Thrust : {result['thrust'].to('kN').value:.1f} kN")
print(f"Isp    : {result['Isp'].value:.1f} s")
print(f"V_exit : {result['V_exit'].to('km/s').value:.2f} km/s")

## 3. SweepResult — HTML table with numeric columns

A `SweepResult` renders as a multi-column HTML table. Each row is one sweep point.

In [ ]:
# Parametric sweep — chamber pressure
sweep = nozzle.sweep('P0', np.linspace(5e6, 30e6, 8), parallel=4)
sweep  # Renders as HTML table

In [ ]:
# Filter to specific outputs
from IPython.display import HTML
HTML(sweep._repr_html_())  # Same as just typing `sweep` in a cell

In [ ]:
# Extract array for plotting
import matplotlib.pyplot as plt

P0_arr    = np.array([float(v.si) if hasattr(v, 'si') else float(v) for v in sweep['P0']]) / 1e6
thrust_arr = np.array([float(v.si) if hasattr(v, 'si') else float(v) for v in sweep['thrust']]) / 1e3
Isp_arr    = np.array([float(v.si) if hasattr(v, 'si') else float(v) for v in sweep['Isp']])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(P0_arr, thrust_arr, 'b-o', linewidth=2)
ax1.set_xlabel('Chamber Pressure (MPa)')
ax1.set_ylabel('Thrust (kN)')
ax1.set_title('Thrust vs Chamber Pressure')
ax1.grid(True, alpha=0.3)

ax2.plot(P0_arr, Isp_arr, 'r-s', linewidth=2)
ax2.set_xlabel('Chamber Pressure (MPa)')
ax2.set_ylabel('Isp (s)')
ax2.set_title('Isp vs Chamber Pressure')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. SensitivityResult — HTML bar chart

A `SensitivityResult` renders as a bar chart in HTML (no matplotlib required).

In [ ]:
sens = nozzle.sensitivity(outputs=['thrust', 'Isp'])
sens  # Renders as HTML bar chart in Jupyter

In [ ]:
# Top drivers
print('Top 5 drivers of Isp:')
for inp, val in sens.top('Isp', n=5):
    bar = '#' * int(abs(val) * 20)
    print(f'  {inp:20s} {val:+.4f}  {bar}')

## 5. Combined workflow — heat exchanger

Coupled system with Gauss-Seidel solver, monitoring, and sweep.

In [ ]:
def ntu_effectiveness(NTU, C_ratio):
    eps = (1 - np.exp(-NTU*(1 - C_ratio))) / (1 - C_ratio*np.exp(-NTU*(1 - C_ratio)))
    return {'effectiveness': eps}

def compute_ntu_and_cratio(UA, C_hot, C_cold):
    C_min = min(C_hot, C_cold)
    C_max = max(C_hot, C_cold)
    return {'NTU': UA / C_min, 'C_min': C_min, 'C_ratio': C_min / C_max}

def heat_balance(effectiveness, C_min, T_hot_in, T_cold_in):
    Q_dot = effectiveness * C_min * (T_hot_in - T_cold_in)
    return {'Q_dot': Q(Q_dot, 'W')}

def outlet_temps(Q_dot, C_hot, C_cold, T_hot_in, T_cold_in):
    T_hot_out  = T_hot_in  - Q_dot / C_hot
    T_cold_out = T_cold_in + Q_dot / C_cold
    return {'T_hot_out':  Q(T_hot_out,  'K'),
            'T_cold_out': Q(T_cold_out, 'K')}

hx = System('counter_flow_hx')
hx.add('T_hot_in',  600, 'K')
hx.add('T_cold_in', 300, 'K')
hx.add('C_hot',    2000)    # W/K
hx.add('C_cold',   3000)    # W/K
hx.add('UA',       5000)    # W/K

hx.use(compute_ntu_and_cratio)
hx.use(ntu_effectiveness)
hx.use(heat_balance)
hx.use(outlet_temps)

hx_result = hx.solve_gauss_seidel(relaxation=0.7, max_iter=200, monitor=False)
hx_result

In [ ]:
# Sweep UA — parallel execution
sweep_ua = hx.sweep('UA', np.linspace(500, 8000, 10), parallel=4,
                    method='gauss_seidel', relaxation=0.7)
sweep_ua

## 6. ISA Atmosphere + Aircraft Performance

In [ ]:
# ISA atmosphere at multiple altitudes
isa_sys = System('isa_sweep')
isa_sys.add('h', 0, 'm')
isa_sys.use('isa_atmosphere')

sweep_isa = isa_sys.sweep('h', np.linspace(0, 20000, 9))
sweep_isa

## 7. Registry — browse built-in RSQs

In [ ]:
# List all built-in RSQs by domain
anvil.registry.list(domain='controls')

In [ ]:
anvil.registry.list(domain='materials')

In [ ]:
# Fuzzy search
anvil.registry.search('fatigue')

In [ ]:
# Detailed info on a specific RSQ
anvil.registry.info('second_order_metrics')

## 8. Unit system demo

In [ ]:
from anvil import K, Pa, m, s, kg, J, kN, km, MPa, in_

# Stub syntax
T_chamber = 3500 * K
P_chamber = 20 * MPa
mdot      = Q(2.5, 'kg/s')

# Arithmetic
F = 150 * kN
print(f'F = {F.value} {F.unit}')
print(f'F = {F.to("N").value:.0f} N')
print(f'F = {F.to("lbf").value:.0f} lbf')

# `in` as inches (Python keyword workaround)
chord = 5.5 * in_
print(f'Chord: {5.5} in = {chord.to("m").value:.4f} m = {chord.to("cm").value:.3f} cm')

# Dimension propagation
KE = 0.5 * Q(1000, 'kg') * Q(250, 'm/s')**2
print(f'KE = {KE.value:.0f} J = {KE.to("kJ").value:.1f} kJ')